# Pandas — Filling NaN Values

`fillna` accepts a **value**, not a function. So filling with mean/median/mode requires computing the value first.

```
Numeric with normal distribution     →  MEAN
Numeric with skew / outliers          →  MEDIAN (robust)
Categorical                            →  MODE (most common)
Time series                            →  FFILL / BFILL
Group-aware                            →  groupby + transform
Custom logic                            →  apply / where
```

In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'name':    ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'salary':  [50000, 60000, 75000, 90000, np.nan],
    'city':    ['NYC', 'LA', 'NYC', 'SF', None],
    'rating':  [4.5, np.nan, 3.8, 5.0, 4.2]
})
df

,name,salary,city,rating
0,Alice,50000.0,NYC,4.5
1,Bob,60000.0,LA,NaN
2,Carol,75000.0,NYC,3.8
3,Dave,90000.0,SF,5.0
4,Eve,NaN,None,4.2


## 1 — Fill with Mean (numeric, normal-ish distribution)

In [3]:
df['salary'].fillna(df['salary'].mean())

0    50000.0
1    60000.0
2    75000.0
3    90000.0
4    68750.0
Name: salary, dtype: float64

## 2 — Fill with Median (numeric, skewed or has outliers)

Median is more robust than mean — one extreme outlier won't drag it.

In [4]:
df['salary'].fillna(df['salary'].median())

0    50000.0
1    60000.0
2    75000.0
3    90000.0
4    67500.0
Name: salary, dtype: float64

## 3 — Fill with Mode (categorical / most common)

`mode()` returns a **Series** (because there might be multiple modes) — take `[0]`.

In [8]:
# Mode returns a Series — take first element
df['city'].mode()[0]

'NYC'

In [6]:
df['city'].fillna(df['city'].mode()[0])

0    NYC
1     LA
2    NYC
3     SF
4    NYC
Name: city, dtype: object

## 4 — Fill Multiple Columns Differently (dict)

Pass a dict to fill each column with its own value:

In [9]:
df.fillna({
    'salary': df['salary'].median(),      # numeric → median
    'city':    df['city'].mode()[0],       # categorical → mode
    'rating':  0                            # specific → 0
})

,name,salary,city,rating
0,Alice,50000.0,NYC,4.5
1,Bob,60000.0,LA,0.0
2,Carol,75000.0,NYC,3.8
3,Dave,90000.0,SF,5.0
4,Eve,67500.0,NYC,4.2


## 5 — Custom Lambda via `apply`

`fillna` itself doesn't take a function. Use `apply` instead.

In [10]:
# Custom logic: NaN → 0, otherwise keep value
df['salary'].apply(lambda x: 0 if pd.isna(x) else x)

0    50000.0
1    60000.0
2    75000.0
3    90000.0
4        0.0
Name: salary, dtype: float64

In [11]:
# More sophisticated: NaN → mean × random factor
import random
mean_salary = df['salary'].mean()
df['salary'].apply(lambda x: round(mean_salary * random.uniform(0.9, 1.1)) if pd.isna(x) else x)

0    50000.0
1    60000.0
2    75000.0
3    90000.0
4    67789.0
Name: salary, dtype: float64

## 6 — Custom Logic via `where`

`where(condition, replacement)` keeps original where condition is True, otherwise uses replacement.

In [12]:
df['salary']

0    50000.0
1    60000.0
2    75000.0
3    90000.0
4        NaN
Name: salary, dtype: float64

In [13]:
# Where NaN, replace with the median
df['salary'].where(df['salary'].notna(), df['salary'].median())

0    50000.0
1    60000.0
2    75000.0
3    90000.0
4    67500.0
Name: salary, dtype: float64

## 7 — Group-Aware Fill (most powerful)

Fill NaN with the **group's** statistic — e.g., each person gets their city's average salary.

In [14]:
# Fill NaN salary with the MEAN salary of that person's city
df['salary_grouped'] = df.groupby('city')['salary'].transform(
    lambda x: x.fillna(x.mean())
)
df

,name,salary,city,rating,salary_grouped
0,Alice,50000.0,NYC,4.5,50000.0
1,Bob,60000.0,LA,NaN,60000.0
2,Carol,75000.0,NYC,3.8,75000.0
3,Dave,90000.0,SF,5.0,90000.0
4,Eve,NaN,None,4.2,NaN


`transform` preserves the original shape — each NaN gets replaced with the group statistic of its row.

## 8 — Forward Fill / Backward Fill (time series)

`ffill` carries the last valid value forward. `bfill` propagates the next valid value backward.

In [15]:
ts = pd.Series([10, np.nan, np.nan, 20, np.nan, 30])

print('Original:    ', ts.tolist())
print('Forward fill:', ts.ffill().tolist())
print('Back fill:   ', ts.bfill().tolist())

Original:     [10.0, nan, nan, 20.0, nan, 30.0]
Forward fill: [10.0, 10.0, 10.0, 20.0, 20.0, 30.0]
Back fill:    [10.0, 20.0, 20.0, 20.0, 30.0, 30.0]


Useful for time series — carry the last known sensor reading, stock price, etc.

## 9 — Interpolation (linear interpolation between values)

In [16]:
ts.interpolate()    # linear interpolation between 10 → 20 → 30

0    10.000000
1    13.333333
2    16.666667
3    20.000000
4    25.000000
5    30.000000
dtype: float64

Linear interpolation: NaN values get values along the straight line between neighbours.

```
10 → ? → ? → 20 → ? → 30
10 → 13.33 → 16.66 → 20 → 25 → 30
```

## 10 — Bulk Fill by Column Type

Fill all numeric with median, all categorical with mode — automatically.

In [17]:
# All numeric columns → median
numeric_cols = df.select_dtypes(include='number').columns
for c in numeric_cols:
    df[c] = df[c].fillna(df[c].median())

# All categorical columns → mode
cat_cols = df.select_dtypes(include='object').columns
for c in cat_cols:
    if df[c].mode().size > 0:    # mode exists
        df[c] = df[c].fillna(df[c].mode()[0])

df

,name,salary,city,rating,salary_grouped
0,Alice,50000.0,NYC,4.50,50000.0
1,Bob,60000.0,LA,4.35,60000.0
2,Carol,75000.0,NYC,3.80,75000.0
3,Dave,90000.0,SF,5.00,90000.0
4,Eve,67500.0,NYC,4.20,67500.0


## When to Use Which Strategy

| Data type | Strategy |
|-----------|----------|
| Numeric, normal distribution | **Mean** |
| Numeric, skewed / outliers | **Median** |
| Categorical | **Mode** |
| Time series | **ffill / bfill / interpolate** |
| Per-group context matters | **groupby + transform** |
| Custom logic needed | **apply / where** |
| Domain-specific default | **specific constant** (0, 'unknown', -1) |

## Caveats

```
⚠ Filling can BIAS your analysis
   Filling salary NaN with 0 → drops the mean substantially

⚠ Filling with mean REDUCES variance
   → can underestimate std dev, correlation

⚠ Sometimes "missingness" itself is informative
   Consider adding a 'was_missing' flag column

⚠ Many ML models handle NaN natively
   XGBoost, LightGBM, CatBoost don't need imputation
```

**Always think about what NaN means in your data before filling.**

## Summary

```
Mean:    df['col'].fillna(df['col'].mean())
Median:  df['col'].fillna(df['col'].median())
Mode:    df['col'].fillna(df['col'].mode()[0])     ← mode returns Series, take [0]

Custom (lambda via apply):
   df['col'].apply(lambda x: <expr> if pd.isna(x) else x)

Custom (where):
   df['col'].where(df['col'].notna(), <value>)

Group-aware:
   df.groupby('group')['col'].transform(lambda x: x.fillna(x.mean()))

Time series:
   df['col'].ffill()           ← forward fill
   df['col'].bfill()           ← backward fill
   df['col'].interpolate()     ← linear interpolation

Multi-column:
   df.fillna({'col1': v1, 'col2': v2, ...})
```